# Live demo: Агрегатор ↔ Эксплуатант

Этот notebook имитирует работу **Агрегатора**, общаясь с **запущенным** Эксплуатантом (Operator) через брокер сообщений.

- Поддерживаемые транспорты: **Mosquitto/MQTT** и **Kafka** (переключается в ноутбуке).
- Тип взаимодействия: request/response по `SystemBus`.

## Предусловия

Поднять Эксплуатанта в Docker:

- MQTT (default):

```bash
docker compose -f systems/operator/docker-compose.yml up -d --build
```

- Kafka (опционально):

```bash
cd systems/operator && make up-kafka
```

## Сценарий

1. Агрегатор отправляет заказ (`receive_order`), получает предложение (цена/срок/параметры)
2. Агрегатор подтверждает выбор Эксплуатанта (`submit_proposal`, затем `accept_order`)
3. Запуск выполнения (`start_mission`) и проверка статуса (`get_mission_status`)
4. Завершение (`complete_mission`) и финальный статус


## Диаграммы взаимодействия

### Диаграмма 1: жизненный цикл заказа

```mermaid
sequenceDiagram
  participant Customer
  participant Aggregator
  participant Operator

  Customer->>Aggregator: CreateOrder
  Aggregator->>Operator: receive_order(order)
  Operator-->>Aggregator: proposal(price,delivery_time,...)

  Aggregator->>Operator: submit_proposal(order_id)
  Operator-->>Aggregator: submitted

  Aggregator->>Operator: accept_order(order_id)
  Operator-->>Aggregator: accepted(mission_id,uas_id)

  Aggregator->>Operator: start_mission(mission_id)
  Operator-->>Aggregator: started

  loop UntilCompleted
    Aggregator->>Operator: get_mission_status(mission_id)
    Operator-->>Aggregator: status
  end

  Aggregator->>Operator: complete_mission(mission_id)
  Operator-->>Aggregator: completed
```

### Диаграмма 2: envelope request/response

```mermaid
sequenceDiagram
  participant Aggregator
  participant Bus
  participant Operator

  Aggregator->>Bus: request(topic, {action,payload,sender})
  Bus->>Operator: publish(topic, message+{correlation_id,reply_to})
  Operator->>Bus: publish(reply_to, response)
  Bus-->>Aggregator: response({correlation_id,payload,success,error?})
```


In [ ]:
# Настройки (выберите брокер: 'mqtt' или 'kafka')
import os
import sys
from pathlib import Path

# В VSCode notebook обычно cwd=notebooks/.
# Добавляем корень репозитория в sys.path, чтобы работали импорты `broker`, `systems`, `sdk`.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "broker").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

broker = os.getenv("AGG_BROKER", "mqtt")  # "mqtt" | "kafka"

SYSTEM_ID = os.getenv("SYSTEM_ID", "operator-001")
API_VERSION = os.getenv("API_VERSION", "v1")

# MQTT
MQTT_BROKER = os.getenv("MQTT_BROKER", "localhost")
MQTT_PORT = int(os.getenv("MQTT_PORT", "1883"))

# Kafka (для ноутбука используем внешний listener)
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092")

OPERATOR_TOPIC = f"{SYSTEM_ID}.{API_VERSION}.operator"

print("repo_root=", repo_root)
print("broker=", broker)
print("OPERATOR_TOPIC=", OPERATOR_TOPIC)
print("MQTT=", f"{MQTT_BROKER}:{MQTT_PORT}")
print("KAFKA=", KAFKA_BOOTSTRAP_SERVERS)


In [ ]:
import time
from dataclasses import dataclass


@dataclass
class BusConfig:
    broker: str
    bus: object


def create_bus(selected: str) -> BusConfig:
    selected = selected.lower().strip()

    if selected == "mqtt":
        from broker.mqtt.mqtt_system_bus import MQTTSystemBus

        b = MQTTSystemBus(broker=MQTT_BROKER, port=MQTT_PORT, client_id=f"aggregator.{SYSTEM_ID}")
        b.start()
        return BusConfig(broker="mqtt", bus=b)

    if selected == "kafka":
        from broker.kafka.kafka_system_bus import KafkaSystemBus

        b = KafkaSystemBus(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            client_id=f"aggregator.{SYSTEM_ID}",
            group_id=f"aggregator-{SYSTEM_ID}",
        )
        b.start()
        return BusConfig(broker="kafka", bus=b)

    raise ValueError(f"Unknown broker: {selected}")


bus_cfg = create_bus(broker)
bus = bus_cfg.bus
print("Bus started:", bus_cfg.broker)


In [ ]:
def request(action: str, payload: dict, timeout_s: float = 30.0) -> dict:
    """Aggregator -> Operator request/response."""
    resp = bus.request(
        OPERATOR_TOPIC,
        {
            "action": action,
            "sender": "aggregator-demo",
            "payload": payload,
        },
        timeout=timeout_s,
    )
    if resp is None:
        raise TimeoutError(f"No response for action={action}")
    payload_resp = resp.get("payload", {})
    if "error" in payload_resp:
        raise RuntimeError(f"Operator error: {payload_resp.get('error')} ({payload_resp})")
    return payload_resp


def pretty(d: dict) -> None:
    import json

    print(json.dumps(d, ensure_ascii=False, indent=2))


In [ ]:
# 1) Создаём заказ и получаем предложение
order_id = f"ORDER-LIVE-{int(time.time())}"
order = {
    "id": order_id,
    "pickup": {"lat": 55.76, "lon": 37.62},
    "dropoff": {"lat": 55.75, "lon": 37.61},
    "payload_weight": 3.5,
    "distance_km": 10.0,
    "payload_value": 20000,
}

receive_res = request("receive_order", {"order": order}, timeout_s=30.0)
print("receive_order response")
pretty(receive_res)

proposal = receive_res.get("proposal", {})
print("proposal")
pretty(proposal)


In [ ]:
# 2) Подтверждаем, что предложение отправлено (submit_proposal)
submit_res = request("submit_proposal", {"order_id": order_id}, timeout_s=15.0)
print("submit_proposal response")
pretty(submit_res)

# 3) Выбираем Эксплуатанта исполнителем (accept_order)
accept_res = request("accept_order", {"order_id": order_id}, timeout_s=30.0)
print("accept_order response")
pretty(accept_res)

mission_id = accept_res.get("mission_id")
if not mission_id:
    raise RuntimeError(f"accept_order did not return mission_id: {accept_res}")


In [ ]:
# 4) Запускаем выполнение и опрашиваем статус
start_res = request("start_mission", {"mission_id": mission_id}, timeout_s=30.0)
print("start_mission response")
pretty(start_res)

status = None
for _ in range(10):
    status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
    print("get_mission_status")
    pretty(status)
    if status.get("status") in {"completed", "failed", "aborted"}:
        break
    time.sleep(1.0)

# 5) Завершаем миссию (если ещё не завершена)
if status and status.get("status") != "completed":
    complete_res = request("complete_mission", {"mission_id": mission_id}, timeout_s=30.0)
    print("complete_mission response")
    pretty(complete_res)

final_status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
print("final get_mission_status")
pretty(final_status)


In [ ]:
# Cleanup (по желанию)
# bus.stop()
# print("Bus stopped")
